In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q torch_geometric

import os, json, math, time, copy
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, SAGPooling, global_mean_pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

D = '/content/drive/MyDrive/PROJECT'
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEV)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda


In [ ]:
paket_pos = torch.load(f'{D}/data/processed/pyg/v2_cfg_prune_pos.pt', weights_only=False)
data_train_pos = paket_pos['train']
data_bench_pos = paket_pos['benchmark']
DIM_POS = paket_pos['dim']

y_pos = np.array([int(d.y) for d in data_train_pos])
print(f'train: {len(data_train_pos)} graf, dimensi: {DIM_POS}')
print(f'benchmark: {len(data_bench_pos)} graf')

train: 610 graf, dimensi: 23
benchmark: 16 graf


In [ ]:
class MultiScaleGAT_MeanPool(nn.Module):
    def __init__(self, dim_input=22, dim_hidden=512, n_kelas=2, dropout=0.5):
        super().__init__()
        h = dim_hidden
        self.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=dropout)
        self.gat2 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.gat3 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.proj = nn.Linear(dim_input, h)
        self.norm1 = nn.LayerNorm(h)
        self.norm2 = nn.LayerNorm(h)
        self.norm3 = nn.LayerNorm(h)
        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_kelas),
        )
        self.dropout = dropout

    def encode(self, x, edge_index, batch):
        h0 = self.proj(x)
        h1 = self.norm1(F.elu(self.gat1(x, edge_index)) + h0)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.norm2(F.elu(self.gat2(h1, edge_index)) + h1)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        h3 = self.norm3(F.elu(self.gat3(h2, edge_index)) + h2)
        return global_mean_pool(h3, batch)

    def forward(self, data):
        z = self.encode(data.x, data.edge_index, data.batch)
        return self.classifier(z)

print('arsitektur siap')

arsitektur siap


In [ ]:
def bobot_kelas(dataset):
    y = np.array([int(d.y) for d in dataset])
    n0, n1 = (y == 0).sum(), (y == 1).sum()
    w = torch.tensor([len(y) / (2 * n0), len(y) / (2 * n1)], dtype=torch.float)
    return w.to(DEV)

@torch.no_grad()
def evaluasi(model, loader):
    model.eval()
    ys, preds, probs = [], [], []
    for b in loader:
        b = b.to(DEV)
        logit = model(b)
        p = torch.softmax(logit, dim=1)[:, 1]
        preds += logit.argmax(1).cpu().tolist()
        probs += p.cpu().tolist()
        ys += b.y.cpu().tolist()
    m = {
        'acc': accuracy_score(ys, preds),
        'prec': precision_score(ys, preds, zero_division=0),
        'rec': recall_score(ys, preds, zero_division=0),
        'f1': f1_score(ys, preds, zero_division=0),
    }
    m['delta_pr'] = abs(m['prec'] - m['rec'])
    try:
        m['auc'] = roc_auc_score(ys, probs)
    except Exception:
        m['auc'] = float('nan')
    return m

def latih_satu_fold(tr, va, ModelClass, dim_input, dim_hidden=512, max_epoch=50, sabar=15,
                    lr=1e-3, batch_size=32, verbose=True):
    model = ModelClass(dim_input=dim_input, dim_hidden=dim_hidden).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = torch.nn.CrossEntropyLoss(weight=bobot_kelas(tr))
    ld_tr = DataLoader(tr, batch_size=batch_size, shuffle=True)
    ld_va = DataLoader(va, batch_size=batch_size)
    f1_terbaik, bobot_terbaik, mandek = -1, None, 0
    for ep in range(1, max_epoch + 1):
        model.train()
        total = 0
        for b in ld_tr:
            b = b.to(DEV)
            opt.zero_grad()
            loss = lossf(model(b), b.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * b.num_graphs
        m_va = evaluasi(model, ld_va)
        if verbose and ep % 10 == 0:
            print(f'    ep{ep:3} loss={total/len(tr):.4f} f1_va={m_va["f1"]:.3f}')
        if m_va['f1'] > f1_terbaik:
            f1_terbaik, bobot_terbaik, mandek = m_va['f1'], copy.deepcopy(model.state_dict()), 0
        else:
            mandek += 1
        if mandek >= sabar:
            if verbose:
                print(f'    early stopping di epoch {ep}')
            break
    model.load_state_dict(bobot_terbaik)
    return model, evaluasi(model, ld_va)

print('fungsi siap')

fungsi siap


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
hasil_pos, model_fold_pos = [], []
mulai = time.time()

for k, (i_tr, i_va) in enumerate(skf.split(np.zeros(len(y_pos)), y_pos), 1):
    print(f'\n=== Fold {k}/5  (train {len(i_tr)}, val {len(i_va)}) ===')
    tr = [data_train_pos[i] for i in i_tr]
    va = [data_train_pos[i] for i in i_va]
    model, m = latih_satu_fold(tr, va, MultiScaleGAT_MeanPool, dim_input=DIM_POS)
    m['fold'] = k
    hasil_pos.append(m)
    model_fold_pos.append(model)
    print(f'  F1={m["f1"]:.4f} Acc={m["acc"]:.4f} P={m["prec"]:.4f} R={m["rec"]:.4f} AUC={m["auc"]:.4f}')

hf_pos = pd.DataFrame(hasil_pos)
print(f'\n=== Rata-rata 5 fold ({(time.time()-mulai)/60:.1f} menit) ===')
print(hf_pos[['acc','prec','rec','f1','delta_pr','auc']].agg(['mean','std']).round(4))

os.makedirs(f'{D}/hasil', exist_ok=True)
hf_pos.to_csv(f'{D}/hasil/kfold_v2_cfg_prune_pos.csv', index=False)
torch.save({'state_dicts': [m.state_dict() for m in model_fold_pos],
            'hasil': hasil_pos, 'dim_hidden': 512, 'dim_input': DIM_POS},
           f'{D}/hasil/model_v2_cfg_prune_pos.pt')
print('tersimpan ke', f'{D}/hasil/')


=== Fold 1/5  (train 488, val 122) ===
    ep 10 loss=0.3087 f1_va=0.869
    ep 20 loss=0.2777 f1_va=0.889
    ep 30 loss=0.2609 f1_va=0.918
    early stopping di epoch 33
  F1=0.9263 Acc=0.9426 P=0.9362 R=0.9167 AUC=0.9651

=== Fold 2/5  (train 488, val 122) ===
    ep 10 loss=0.3299 f1_va=0.791
    ep 20 loss=0.2683 f1_va=0.831
    ep 30 loss=0.2586 f1_va=0.875
    ep 40 loss=0.2276 f1_va=0.896
    early stopping di epoch 40
  F1=0.8980 Acc=0.9180 P=0.8980 R=0.8980 AUC=0.9536

=== Fold 3/5  (train 488, val 122) ===
    ep 10 loss=0.3318 f1_va=0.782
    ep 20 loss=0.2571 f1_va=0.786
    early stopping di epoch 28
  F1=0.8222 Acc=0.8689 P=0.9024 R=0.7551 AUC=0.9416

=== Fold 4/5  (train 488, val 122) ===
    ep 10 loss=0.3166 f1_va=0.833
    ep 20 loss=0.3111 f1_va=0.800
    early stopping di epoch 26
  F1=0.8542 Acc=0.8852 P=0.8723 R=0.8367 AUC=0.9284

=== Fold 5/5  (train 488, val 122) ===
    ep 10 loss=0.3132 f1_va=0.817
    ep 20 loss=0.2876 f1_va=0.813
    early stopping di epoc

In [ ]:
simpan_pos = torch.load(f'{D}/hasil/model_v2_cfg_prune_pos.pt', weights_only=False)
models_pos = []
for sd in simpan_pos['state_dicts']:
    m = MultiScaleGAT_MeanPool(dim_input=DIM_POS, dim_hidden=simpan_pos['dim_hidden']).to(DEV)
    m.load_state_dict(sd)
    m.eval()
    models_pos.append(m)

ld = DataLoader(data_bench_pos, batch_size=16)
prob_semua_pos = []
with torch.no_grad():
    for m in models_pos:
        p = []
        for b in ld:
            b = b.to(DEV)
            p += torch.softmax(m(b), dim=1)[:, 1].cpu().tolist()
        prob_semua_pos.append(p)
prob_semua_pos = np.array(prob_semua_pos)
prob_rata_pos = prob_semua_pos.mean(axis=0)

tabel_pos = pd.DataFrame({
    'nama': [d.nama for d in data_bench_pos],
    'label': [int(d.y) for d in data_bench_pos],
    'prob_rentan': prob_rata_pos.round(3),
    'pred': (prob_rata_pos >= 0.5).astype(int),
})
tabel_pos['benar'] = tabel_pos.label == tabel_pos.pred
print(tabel_pos.sort_values(['label','nama']).to_string(index=False))

y_b, pr_b = tabel_pos.label, tabel_pos.pred
print(f'\nAcc={accuracy_score(y_b,pr_b):.4f}  P={precision_score(y_b,pr_b,zero_division=0):.4f}  '
      f'R={recall_score(y_b,pr_b,zero_division=0):.4f}  F1={f1_score(y_b,pr_b,zero_division=0):.4f}  '
      f'AUC={roc_auc_score(y_b,tabel_pos.prob_rentan):.4f}')

print('\n=== Analisis per pasangan (fitur posisi) ===')
cari_pos = dict(zip(tabel_pos.nama, tabel_pos.prob_rentan))
lolos_pos = 0
for f in sorted(tabel_pos[tabel_pos.label==1].nama):
    r = cari_pos[f]
    a = cari_pos.get(f.replace('.sol','_fixed.sol'))
    ok = r > a
    lolos_pos += ok
    print(f'{f:28} rentan={r:.3f}  aman={a:.3f}  {"OK" if ok else "TERBALIK"}')
print(f'\n{lolos_pos}/8 pasangan diurutkan dengan benar')

                         nama  label  prob_rentan  pred  benar
          etherbank_fixed.sol      0        0.000     0   True
         etherstore_fixed.sol      0        0.000     0   True
modifier_reentrancy_fixed.sol      0        0.000     0   True
         reentrance_fixed.sol      0        0.115     0   True
     reentrancy_dao_fixed.sol      0        0.200     0   True
reentrancy_insecure_fixed.sol      0        0.131     0   True
  reentrancy_simple_fixed.sol      0        0.000     0   True
         simple_dao_fixed.sol      0        0.202     0   True
                etherbank.sol      1        0.000     0  False
               etherstore.sol      1        0.000     0  False
      modifier_reentrancy.sol      1        0.037     0  False
               reentrance.sol      1        0.098     0  False
           reentrancy_dao.sol      1        0.241     0  False
      reentrancy_insecure.sol      1        0.600     1   True
        reentrancy_simple.sol      1        0.001     0

## YANG LAMA2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q torch_geometric

import os, json, math, time, copy
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, SAGPooling, global_mean_pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

D = '/content/drive/MyDrive/PROJECT'
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEV)

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 58.4 MB/s eta 0:00:00
device: cuda


In [ ]:
OUT_PRUNE = f'{D}/data/processed/v2_cfg_prune_0.02'
n = len([f for f in os.listdir(OUT_PRUNE) if f.endswith('.json')])
print(f'{n} file graf sumber (v2, sebelum fitur posisi)')

627 file graf sumber (v2, sebelum fitur posisi)


In [ ]:
class MultiScaleGAT_MeanPool(nn.Module):
    def __init__(self, dim_input=22, dim_hidden=512, n_kelas=2, dropout=0.5):
        super().__init__()
        h = dim_hidden
        self.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=dropout)
        self.gat2 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.gat3 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.proj = nn.Linear(dim_input, h)
        self.norm1 = nn.LayerNorm(h)
        self.norm2 = nn.LayerNorm(h)
        self.norm3 = nn.LayerNorm(h)
        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_kelas),
        )
        self.dropout = dropout

    def encode(self, x, edge_index, batch):
        h0 = self.proj(x)
        h1 = self.norm1(F.elu(self.gat1(x, edge_index)) + h0)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.norm2(F.elu(self.gat2(h1, edge_index)) + h1)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        h3 = self.norm3(F.elu(self.gat3(h2, edge_index)) + h2)
        return global_mean_pool(h3, batch)

    def forward(self, data):
        z = self.encode(data.x, data.edge_index, data.batch)
        return self.classifier(z)

print('arsitektur siap')

arsitektur siap


In [ ]:
from collections import deque

def tambah_fitur_posisi(g):
    """Tambahkan jarak topologis dari entrypoint (dinormalisasi 0-1) ke tiap node."""
    keluar = {}
    for a, b, t in g['edges']:
        if t == 0:  # hanya edge alur eksekusi intra-fungsi
            keluar.setdefault(a, []).append(b)

    entrypoints = [n['id'] for n in g['nodes'] if n['tipe'] in ('ENTRYPOINT','OTHER_ENTRYPOINT')]
    jarak = {}
    for ep in entrypoints:
        q, terlihat = deque([(ep, 0)]), {ep}
        while q:
            cur, d = q.popleft()
            if cur not in jarak or d < jarak[cur]:
                jarak[cur] = d
            for nxt in keluar.get(cur, []):
                if nxt not in terlihat:
                    terlihat.add(nxt)
                    q.append((nxt, d + 1))

    max_jarak = max(jarak.values()) if jarak else 1
    max_jarak = max(max_jarak, 1)
    for n in g['nodes']:
        n['jarak_entry'] = jarak.get(n['id'], 0) / max_jarak

    return g

print('fungsi fitur posisi siap')

fungsi fitur posisi siap


In [ ]:
SUMBER = f'{D}/data/processed/v2_cfg_prune_0.02'
TUJUAN = f'{D}/data/processed/v2_cfg_prune_pos'
os.makedirs(TUJUAN, exist_ok=True)

mulai = time.time()
n_proses = 0
for f in sorted(os.listdir(SUMBER)):
    if not f.endswith('.json'):
        continue
    with open(f'{SUMBER}/{f}') as fh:
        g = json.load(fh)
    g = tambah_fitur_posisi(g)
    with open(f'{TUJUAN}/{f}', 'w') as fh:
        json.dump(g, fh)
    n_proses += 1

print(f'{n_proses} graf diproses dalam {time.time()-mulai:.1f} detik')
print(f'tersimpan di: {TUJUAN}')

# uji cepat: lihat sebaran jarak_entry di satu kontrak
with open(f'{TUJUAN}/3_expert_benchmark__rentan__simple_dao.json') as f:
    contoh = json.load(f)
for n in sorted(contoh['nodes'], key=lambda x: x['id']):
    print(f"  node {n['id']} {n['tipe']:12} jarak_entry={n['jarak_entry']:.2f}")

KeyboardInterrupt: 

In [ ]:
TIPE = ['EXPRESSION','ENTRYPOINT','RETURN','VARIABLE','IF','ENDIF',
        'OTHER_ENTRYPOINT','STARTLOOP','ENDLOOP','IFLOOP','ASSEMBLY',
        'ENDASSEMBLY','BREAK','CONTINUE','CATCH','TRY','THROW']
TIPE_IDX = {t: i for i, t in enumerate(TIPE)}
DIM_POS = len(TIPE) + 6   # 17 tipe + 5 fitur numerik lama + 1 jarak_entry = 23

def vektor_node_pos(n):
    v = [0.0] * DIM_POS
    idx = TIPE_IDX.get(n['tipe'])
    if idx is not None:
        v[idx] = 1.0
    o = len(TIPE)
    v[o+0] = math.log1p(n['n_call_low'])
    v[o+1] = math.log1p(n['n_call_high'])
    v[o+2] = math.log1p(n['n_sstore'])
    v[o+3] = math.log1p(n['n_sload'])
    v[o+4] = math.log1p(n['n_irs'])
    v[o+5] = n.get('jarak_entry', 0.0)
    return v

def ke_pyg_pos(g):
    nodes = sorted(g['nodes'], key=lambda x: x['id'])
    x = torch.tensor([vektor_node_pos(n) for n in nodes], dtype=torch.float)
    if g['edges']:
        ei = torch.tensor([[e[0] for e in g['edges']], [e[1] for e in g['edges']]], dtype=torch.long)
        et = torch.tensor([e[2] for e in g['edges']], dtype=torch.long)
    else:
        ei = torch.empty((2,0), dtype=torch.long)
        et = torch.empty((0,), dtype=torch.long)
    return Data(x=x, edge_index=ei, edge_type=et,
                y=torch.tensor([g['label']], dtype=torch.long),
                nama=g['nama_file'], split=g['split'], folder=g['folder'])

TUJUAN = f'{D}/data/processed/v2_cfg_prune_pos'
OUT_PYG = f'{D}/data/processed/pyg'

train_pos, bench_pos, kosong = [], [], []
for f in sorted(os.listdir(TUJUAN)):
    if not f.endswith('.json'):
        continue
    with open(f'{TUJUAN}/{f}') as fh:
        g = json.load(fh)
    split = 'train' if g['folder'] in ('0_normal','1_reentrancy') else 'benchmark'
    g['split'] = split
    d = ke_pyg_pos(g)
    if d.num_nodes == 0 or d.edge_index.size(1) == 0:
        kosong.append(g['nama_file'])
        continue
    (train_pos if split=='train' else bench_pos).append(d)

torch.save({'train': train_pos, 'benchmark': bench_pos, 'dim': DIM_POS, 'tipe': TIPE},
           f'{OUT_PYG}/v2_cfg_prune_pos.pt')

import collections
print(f'train    : {len(train_pos)} graf, label={collections.Counter(int(d.y) for d in train_pos)}')
print(f'benchmark: {len(bench_pos)} graf, label={collections.Counter(int(d.y) for d in bench_pos)}')
print(f'dilewati : {len(kosong)}')
print(f'dimensi  : {DIM_POS}')
print(f'tersimpan: {OUT_PYG}/v2_cfg_prune_pos.pt')

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
def bobot_kelas(dataset):
    y = np.array([int(d.y) for d in dataset])
    n0, n1 = (y == 0).sum(), (y == 1).sum()
    w = torch.tensor([len(y) / (2 * n0), len(y) / (2 * n1)], dtype=torch.float)
    return w.to(DEV)

@torch.no_grad()
def evaluasi(model, loader):
    model.eval()
    ys, preds, probs = [], [], []
    for b in loader:
        b = b.to(DEV)
        logit = model(b)
        p = torch.softmax(logit, dim=1)[:, 1]
        preds += logit.argmax(1).cpu().tolist()
        probs += p.cpu().tolist()
        ys += b.y.cpu().tolist()
    m = {
        'acc': accuracy_score(ys, preds),
        'prec': precision_score(ys, preds, zero_division=0),
        'rec': recall_score(ys, preds, zero_division=0),
        'f1': f1_score(ys, preds, zero_division=0),
    }
    m['delta_pr'] = abs(m['prec'] - m['rec'])
    try:
        m['auc'] = roc_auc_score(ys, probs)
    except Exception:
        m['auc'] = float('nan')
    return m

def latih_satu_fold(tr, va, ModelClass, dim_input, dim_hidden=512, max_epoch=50, sabar=15,
                    lr=1e-3, batch_size=32, verbose=True):
    model = ModelClass(dim_input=dim_input, dim_hidden=dim_hidden).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = torch.nn.CrossEntropyLoss(weight=bobot_kelas(tr))
    ld_tr = DataLoader(tr, batch_size=batch_size, shuffle=True)
    ld_va = DataLoader(va, batch_size=batch_size)
    f1_terbaik, bobot_terbaik, mandek = -1, None, 0
    for ep in range(1, max_epoch + 1):
        model.train()
        total = 0
        for b in ld_tr:
            b = b.to(DEV)
            opt.zero_grad()
            loss = lossf(model(b), b.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * b.num_graphs
        m_va = evaluasi(model, ld_va)
        if verbose and ep % 10 == 0:
            print(f'    ep{ep:3} loss={total/len(tr):.4f} f1_va={m_va["f1"]:.3f}')
        if m_va['f1'] > f1_terbaik:
            f1_terbaik, bobot_terbaik, mandek = m_va['f1'], copy.deepcopy(model.state_dict()), 0
        else:
            mandek += 1
        if mandek >= sabar:
            if verbose:
                print(f'    early stopping di epoch {ep}')
            break
    model.load_state_dict(bobot_terbaik)
    return model, evaluasi(model, ld_va)

print('fungsi siap')

fungsi siap


In [ ]:
paket_pos = torch.load(f'{D}/data/processed/pyg/v2_cfg_prune_pos.pt', weights_only=False)
data_train_pos = paket_pos['train']
data_bench_pos = paket_pos['benchmark']
DIM_POS = paket_pos['dim']

y_pos = np.array([int(d.y) for d in data_train_pos])
print(f'device dipakai: {DEV}')
print(f'train: {len(data_train_pos)} graf, dimensi: {DIM_POS}')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
hasil_pos, model_fold_pos = [], []
mulai = time.time()

for k, (i_tr, i_va) in enumerate(skf.split(np.zeros(len(y_pos)), y_pos), 1):
    print(f'\n=== Fold {k}/5  (train {len(i_tr)}, val {len(i_va)}) ===')
    tr = [data_train_pos[i] for i in i_tr]
    va = [data_train_pos[i] for i in i_va]
    model, m = latih_satu_fold(tr, va, MultiScaleGAT_MeanPool, dim_input=DIM_POS)
    m['fold'] = k
    hasil_pos.append(m)
    model_fold_pos.append(model)
    print(f'  F1={m["f1"]:.4f} Acc={m["acc"]:.4f} P={m["prec"]:.4f} R={m["rec"]:.4f} AUC={m["auc"]:.4f}')

hf_pos = pd.DataFrame(hasil_pos)
print(f'\n=== Rata-rata 5 fold ({(time.time()-mulai)/60:.1f} menit) ===')
print(hf_pos[['acc','prec','rec','f1','delta_pr','auc']].agg(['mean','std']).round(4))

os.makedirs(f'{D}/hasil', exist_ok=True)
hf_pos.to_csv(f'{D}/hasil/kfold_v2_cfg_prune_pos.csv', index=False)
torch.save({'state_dicts': [m.state_dict() for m in model_fold_pos],
            'hasil': hasil_pos, 'dim_hidden': 512, 'dim_input': DIM_POS},
           f'{D}/hasil/model_v2_cfg_prune_pos.pt')
print('tersimpan ke', f'{D}/hasil/')

device dipakai: cpu
train: 610 graf, dimensi: 23

=== Fold 1/5  (train 488, val 122) ===
    ep 10 loss=0.3178 f1_va=0.857
    ep 20 loss=0.2759 f1_va=0.907
    ep 30 loss=0.2408 f1_va=0.871
    ep 40 loss=0.2143 f1_va=0.907
    early stopping di epoch 46
  F1=0.9263 Acc=0.9426 P=0.9362 R=0.9167 AUC=0.9659

=== Fold 2/5  (train 488, val 122) ===
    ep 10 loss=0.3456 f1_va=0.836
    ep 20 loss=0.2646 f1_va=0.822
    ep 30 loss=0.2654 f1_va=0.898
    ep 40 loss=0.2332 f1_va=0.844
    early stopping di epoch 49
  F1=0.9072 Acc=0.9262 P=0.9167 R=0.8980 AUC=0.9444

=== Fold 3/5  (train 488, val 122) ===


## YANG LAMA

In [ ]:
DATASET = 'v2_cfg_prune'
paket = torch.load(f'{D}/data/processed/pyg/{DATASET}.pt', weights_only=False)
data_train = paket['train']
data_bench = paket['benchmark']
DIM = paket['dim']

y = np.array([int(d.y) for d in data_train])
print(f'train    : {len(data_train)} graf, label={dict(zip(*np.unique(y, return_counts=True)))}')
print(f'benchmark: {len(data_bench)} graf')
print(f'dimensi  : {DIM}')

train    : 610 graf, label={np.int64(0): np.int64(366), np.int64(1): np.int64(244)}
benchmark: 16 graf
dimensi  : 22


In [ ]:
class MultiScaleGAT(nn.Module):
    def __init__(self, dim_input=22, dim_hidden=512, n_kelas=2, dropout=0.5):
        super().__init__()
        h = dim_hidden
        self.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=dropout)
        self.gat2 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.gat3 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.proj = nn.Linear(dim_input, h)
        self.norm1 = nn.LayerNorm(h)
        self.norm2 = nn.LayerNorm(h)
        self.norm3 = nn.LayerNorm(h)
        self.pool = SAGPooling(h, ratio=0.5)
        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_kelas),
        )
        self.dropout = dropout

    def encode(self, x, edge_index, batch):
        h0 = self.proj(x)
        h1 = self.norm1(F.elu(self.gat1(x, edge_index)) + h0)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.norm2(F.elu(self.gat2(h1, edge_index)) + h1)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        h3 = self.norm3(F.elu(self.gat3(h2, edge_index)) + h2)
        x_p, ei_p, _, batch_p, _, _ = self.pool(h3, edge_index, batch=batch)
        return global_mean_pool(x_p, batch_p)

    def forward(self, data):
        z = self.encode(data.x, data.edge_index, data.batch)
        return self.classifier(z)

print('arsitektur siap')

arsitektur siap


In [ ]:
def bobot_kelas(dataset):
    y = np.array([int(d.y) for d in dataset])
    n0, n1 = (y == 0).sum(), (y == 1).sum()
    w = torch.tensor([len(y) / (2 * n0), len(y) / (2 * n1)], dtype=torch.float)
    return w.to(DEV)

@torch.no_grad()
def evaluasi(model, loader):
    model.eval()
    ys, preds, probs = [], [], []
    for b in loader:
        b = b.to(DEV)
        logit = model(b)
        p = torch.softmax(logit, dim=1)[:, 1]
        preds += logit.argmax(1).cpu().tolist()
        probs += p.cpu().tolist()
        ys += b.y.cpu().tolist()
    m = {
        'acc': accuracy_score(ys, preds),
        'prec': precision_score(ys, preds, zero_division=0),
        'rec': recall_score(ys, preds, zero_division=0),
        'f1': f1_score(ys, preds, zero_division=0),
    }
    m['delta_pr'] = abs(m['prec'] - m['rec'])
    try:
        m['auc'] = roc_auc_score(ys, probs)
    except Exception:
        m['auc'] = float('nan')
    m['cm'] = confusion_matrix(ys, preds).tolist()
    return m

def latih_satu_fold(tr, va, dim_hidden=512, max_epoch=50, sabar=15,
                    lr=1e-3, batch_size=32, verbose=True):
    model = MultiScaleGAT(dim_input=DIM, dim_hidden=dim_hidden).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = torch.nn.CrossEntropyLoss(weight=bobot_kelas(tr))
    ld_tr = DataLoader(tr, batch_size=batch_size, shuffle=True)
    ld_va = DataLoader(va, batch_size=batch_size)

    f1_terbaik, bobot_terbaik, mandek = -1, None, 0
    riwayat = []
    for ep in range(1, max_epoch + 1):
        model.train()
        total = 0
        for b in ld_tr:
            b = b.to(DEV)
            opt.zero_grad()
            loss = lossf(model(b), b.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * b.num_graphs

        m_tr = evaluasi(model, ld_tr)
        m_va = evaluasi(model, ld_va)
        riwayat.append({'epoch': ep, 'loss': total/len(tr), 'f1_train': m_tr['f1'], 'f1_val': m_va['f1']})

        if m_va['f1'] > f1_terbaik:
            f1_terbaik = m_va['f1']
            bobot_terbaik = copy.deepcopy(model.state_dict())
            mandek = 0
        else:
            mandek += 1

        if verbose and (ep % 5 == 0 or ep == 1):
            print(f'    ep{ep:3} loss={total/len(tr):.4f} f1_tr={m_tr["f1"]:.3f} f1_va={m_va["f1"]:.3f}')
        if mandek >= sabar:
            if verbose:
                print(f'    early stopping di epoch {ep}')
            break

    model.load_state_dict(bobot_terbaik)
    return model, riwayat, evaluasi(model, ld_va)

print('fungsi pelatihan siap')

fungsi pelatihan siap


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
hasil, model_fold, riwayat_fold = [], [], []
mulai = time.time()

for k, (i_tr, i_va) in enumerate(skf.split(np.zeros(len(y)), y), 1):
    print(f'\n=== Fold {k}/5  (train {len(i_tr)}, val {len(i_va)}) ===')
    tr = [data_train[i] for i in i_tr]
    va = [data_train[i] for i in i_va]
    model, riwayat, m = latih_satu_fold(tr, va)
    m['fold'] = k
    hasil.append(m)
    model_fold.append(model)
    riwayat_fold.append(riwayat)
    print(f'  F1={m["f1"]:.4f} Acc={m["acc"]:.4f} P={m["prec"]:.4f} R={m["rec"]:.4f} AUC={m["auc"]:.4f}')

hf = pd.DataFrame(hasil)
print(f'\n=== Rata-rata 5 fold ({(time.time()-mulai)/60:.1f} menit) ===')
print(hf[['acc','prec','rec','f1','delta_pr','auc']].agg(['mean','std']).round(4))

os.makedirs(f'{D}/hasil', exist_ok=True)
hf.to_csv(f'{D}/hasil/kfold_{DATASET}.csv', index=False)
torch.save({'state_dicts': [m.state_dict() for m in model_fold],
            'riwayat': riwayat_fold, 'hasil': hasil, 'dim_hidden': 512},
           f'{D}/hasil/model_{DATASET}.pt')
print('tersimpan ke', f'{D}/hasil/')


=== Fold 1/5  (train 488, val 122) ===
    ep  1 loss=0.6828 f1_tr=0.725 f1_va=0.709
    ep  5 loss=0.5195 f1_tr=0.746 f1_va=0.744
    ep 10 loss=0.4688 f1_tr=0.774 f1_va=0.766
    ep 15 loss=0.4075 f1_tr=0.798 f1_va=0.788
    ep 20 loss=0.4059 f1_tr=0.828 f1_va=0.826
    ep 25 loss=0.3833 f1_tr=0.720 f1_va=0.822
    ep 30 loss=0.3938 f1_tr=0.765 f1_va=0.826
    ep 35 loss=0.3183 f1_tr=0.789 f1_va=0.845
    ep 40 loss=0.3096 f1_tr=0.837 f1_va=0.830
    ep 45 loss=0.3304 f1_tr=0.787 f1_va=0.848
    ep 50 loss=0.3075 f1_tr=0.801 f1_va=0.857
  F1=0.8980 Acc=0.9180 P=0.8800 R=0.9167 AUC=0.9696

=== Fold 2/5  (train 488, val 122) ===
    ep  1 loss=0.6503 f1_tr=0.741 f1_va=0.777
    ep  5 loss=0.3352 f1_tr=0.850 f1_va=0.838
    ep 10 loss=0.3701 f1_tr=0.633 f1_va=0.709
    ep 15 loss=0.3596 f1_tr=0.825 f1_va=0.830
    ep 20 loss=0.2761 f1_tr=0.870 f1_va=0.880
    ep 25 loss=0.2667 f1_tr=0.891 f1_va=0.875
    ep 30 loss=0.2583 f1_tr=0.866 f1_va=0.884
    ep 35 loss=0.2586 f1_tr=0.905 f1_va=

In [ ]:
simpan = torch.load(f'{D}/hasil/model_{DATASET}.pt', weights_only=False)
models = []
for sd in simpan['state_dicts']:
    m = MultiScaleGAT(dim_input=DIM, dim_hidden=simpan['dim_hidden']).to(DEV)
    m.load_state_dict(sd)
    m.eval()
    models.append(m)

ld = DataLoader(data_bench, batch_size=16)
prob_semua = []
with torch.no_grad():
    for m in models:
        p = []
        for b in ld:
            b = b.to(DEV)
            p += torch.softmax(m(b), dim=1)[:, 1].cpu().tolist()
        prob_semua.append(p)
prob_semua = np.array(prob_semua)
prob_rata = prob_semua.mean(axis=0)

tabel = pd.DataFrame({
    'nama': [d.nama for d in data_bench],
    'label': [int(d.y) for d in data_bench],
    'prob_rentan': prob_rata.round(3),
    'pred': (prob_rata >= 0.5).astype(int),
})
tabel['benar'] = tabel.label == tabel.pred
print(tabel.sort_values(['label','nama']).to_string(index=False))

y_b, pr_b = tabel.label, tabel.pred
print(f'\nAcc={accuracy_score(y_b,pr_b):.4f}  P={precision_score(y_b,pr_b,zero_division=0):.4f}  '
      f'R={recall_score(y_b,pr_b,zero_division=0):.4f}  F1={f1_score(y_b,pr_b,zero_division=0):.4f}  '
      f'AUC={roc_auc_score(y_b,tabel.prob_rentan):.4f}')

print('\n=== Analisis per pasangan ===')
cari = dict(zip(tabel.nama, tabel.prob_rentan))
lolos = 0
for f in sorted(tabel[tabel.label==1].nama):
    r = cari[f]
    a = cari.get(f.replace('.sol','_fixed.sol'))
    ok = r > a
    lolos += ok
    print(f'{f:28} rentan={r:.3f}  aman={a:.3f}  {"OK" if ok else "TERBALIK"}')
print(f'\n{lolos}/8 pasangan diurutkan dengan benar')

                         nama  label  prob_rentan  pred  benar
          etherbank_fixed.sol      0        0.020     0   True
         etherstore_fixed.sol      0        0.185     0   True
modifier_reentrancy_fixed.sol      0        0.014     0   True
         reentrance_fixed.sol      0        0.084     0   True
     reentrancy_dao_fixed.sol      0        0.511     1  False
reentrancy_insecure_fixed.sol      0        0.616     1  False
  reentrancy_simple_fixed.sol      0        0.020     0   True
         simple_dao_fixed.sol      0        0.129     0   True
                etherbank.sol      1        0.016     0  False
               etherstore.sol      1        0.186     0  False
      modifier_reentrancy.sol      1        0.075     0  False
               reentrance.sol      1        0.006     0  False
           reentrancy_dao.sol      1        0.509     1   True
      reentrancy_insecure.sol      1        0.701     1   True
        reentrancy_simple.sol      1        0.045     0

In [ ]:
from sklearn.metrics import f1_score
for ambang in [0.3, 0.35, 0.4, 0.45, 0.5]:
    pred_baru = (tabel.prob_rentan >= ambang).astype(int)
    f1 = f1_score(tabel.label, pred_baru, zero_division=0)
    print(f'ambang={ambang}  F1={f1:.3f}  prediksi rentan={pred_baru.sum()}/16')

ambang=0.3  F1=0.333  prediksi rentan=4/16
ambang=0.35  F1=0.333  prediksi rentan=4/16
ambang=0.4  F1=0.333  prediksi rentan=4/16
ambang=0.45  F1=0.333  prediksi rentan=4/16
ambang=0.5  F1=0.333  prediksi rentan=4/16


In [ ]:
ukuran = np.array([d.num_nodes for d in data_train])
kecil = ukuran < 20
print(f'{kecil.sum()} graf latih dengan node < 20')

ld_kecil = DataLoader([d for d, k in zip(data_train, kecil) if k], batch_size=16)
with torch.no_grad():
    prob_k = np.mean([torch.softmax(m(b.to(DEV)),1)[:,1].cpu().numpy()
                       for b in ld_kecil for m in models], axis=0) if kecil.sum() else []
y_k = [int(d.y) for d, k in zip(data_train, kecil) if k]
pred_k = (np.array(prob_k) >= 0.5).astype(int) if len(prob_k) else []
if len(y_k):
    print(f'F1 pada graf latih kecil: {f1_score(y_k, pred_k, zero_division=0):.3f}')

14 graf latih dengan node < 20
F1 pada graf latih kecil: 0.800


In [ ]:
class MultiScaleGAT_MeanPool(nn.Module):
    def __init__(self, dim_input=22, dim_hidden=512, n_kelas=2, dropout=0.5):
        super().__init__()
        h = dim_hidden
        self.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=dropout)
        self.gat2 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.gat3 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.proj = nn.Linear(dim_input, h)
        self.norm1 = nn.LayerNorm(h)
        self.norm2 = nn.LayerNorm(h)
        self.norm3 = nn.LayerNorm(h)
        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_kelas),
        )
        self.dropout = dropout

    def encode(self, x, edge_index, batch):
        h0 = self.proj(x)
        h1 = self.norm1(F.elu(self.gat1(x, edge_index)) + h0)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.norm2(F.elu(self.gat2(h1, edge_index)) + h1)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        h3 = self.norm3(F.elu(self.gat3(h2, edge_index)) + h2)
        return global_mean_pool(h3, batch)   # langsung mean pool, tanpa SAGPooling

    def forward(self, data):
        z = self.encode(data.x, data.edge_index, data.batch)
        return self.classifier(z)

print('varian mean pooling siap')

varian mean pooling siap


In [ ]:
def latih_satu_fold_v2(tr, va, ModelClass, dim_hidden=512, max_epoch=50, sabar=15,
                       lr=1e-3, batch_size=32, verbose=False):
    model = ModelClass(dim_input=DIM, dim_hidden=dim_hidden).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = torch.nn.CrossEntropyLoss(weight=bobot_kelas(tr))
    ld_tr = DataLoader(tr, batch_size=batch_size, shuffle=True)
    ld_va = DataLoader(va, batch_size=batch_size)
    f1_terbaik, bobot_terbaik, mandek = -1, None, 0
    for ep in range(1, max_epoch + 1):
        model.train()
        for b in ld_tr:
            b = b.to(DEV)
            opt.zero_grad()
            loss = lossf(model(b), b.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        m_va = evaluasi(model, ld_va)
        if m_va['f1'] > f1_terbaik:
            f1_terbaik, bobot_terbaik, mandek = m_va['f1'], copy.deepcopy(model.state_dict()), 0
        else:
            mandek += 1
        if mandek >= sabar:
            break
    model.load_state_dict(bobot_terbaik)
    return model, evaluasi(model, ld_va)

models_mp, hasil_mp = [], []
for k, (i_tr, i_va) in enumerate(skf.split(np.zeros(len(y)), y), 1):
    tr = [data_train[i] for i in i_tr]
    va = [data_train[i] for i in i_va]
    model, m = latih_satu_fold_v2(tr, va, MultiScaleGAT_MeanPool)
    hasil_mp.append(m)
    models_mp.append(model)
    print(f'Fold {k}: F1={m["f1"]:.4f}')

print('\nrata-rata F1 (mean pool):', round(np.mean([m['f1'] for m in hasil_mp]), 4))

Fold 1: F1=0.9072
Fold 2: F1=0.9200
Fold 3: F1=0.8980
Fold 4: F1=0.8511
Fold 5: F1=0.8889

rata-rata F1 (mean pool): 0.893


In [ ]:
ld = DataLoader(data_bench, batch_size=16)
prob_semua_mp = []
with torch.no_grad():
    for m in models_mp:
        p = []
        for b in ld:
            b = b.to(DEV)
            p += torch.softmax(m(b), dim=1)[:, 1].cpu().tolist()
        prob_semua_mp.append(p)
prob_semua_mp = np.array(prob_semua_mp)
prob_rata_mp = prob_semua_mp.mean(axis=0)

tabel_mp = pd.DataFrame({
    'nama': [d.nama for d in data_bench],
    'label': [int(d.y) for d in data_bench],
    'prob_rentan': prob_rata_mp.round(3),
    'pred': (prob_rata_mp >= 0.5).astype(int),
})
tabel_mp['benar'] = tabel_mp.label == tabel_mp.pred
print(tabel_mp.sort_values(['label','nama']).to_string(index=False))

y_b, pr_b = tabel_mp.label, tabel_mp.pred
print(f'\nAcc={accuracy_score(y_b,pr_b):.4f}  P={precision_score(y_b,pr_b,zero_division=0):.4f}  '
      f'R={recall_score(y_b,pr_b,zero_division=0):.4f}  F1={f1_score(y_b,pr_b,zero_division=0):.4f}  '
      f'AUC={roc_auc_score(y_b,tabel_mp.prob_rentan):.4f}')

print('\n=== Analisis per pasangan (mean pooling) ===')
cari_mp = dict(zip(tabel_mp.nama, tabel_mp.prob_rentan))
lolos_mp = 0
for f in sorted(tabel_mp[tabel_mp.label==1].nama):
    r = cari_mp[f]
    a = cari_mp.get(f.replace('.sol','_fixed.sol'))
    ok = r > a
    lolos_mp += ok
    print(f'{f:28} rentan={r:.3f}  aman={a:.3f}  {"OK" if ok else "TERBALIK"}')
print(f'\n{lolos_mp}/8 pasangan diurutkan dengan benar')

                         nama  label  prob_rentan  pred  benar
          etherbank_fixed.sol      0        0.002     0   True
         etherstore_fixed.sol      0        0.001     0   True
modifier_reentrancy_fixed.sol      0        0.003     0   True
         reentrance_fixed.sol      0        0.157     0   True
     reentrancy_dao_fixed.sol      0        0.808     1  False
reentrancy_insecure_fixed.sol      0        0.434     0   True
  reentrancy_simple_fixed.sol      0        0.002     0   True
         simple_dao_fixed.sol      0        0.056     0   True
                etherbank.sol      1        0.001     0  False
               etherstore.sol      1        0.001     0  False
      modifier_reentrancy.sol      1        0.447     0  False
               reentrance.sol      1        0.129     0  False
           reentrancy_dao.sol      1        0.933     1   True
      reentrancy_insecure.sol      1        0.790     1   True
        reentrancy_simple.sol      1        0.017     0

# SHAP

In [ ]:
import itertools
from math import factorial

@torch.no_grad()
def prob_rentan_masked(models, data, mask):
    """mask: array 0/1 sepanjang n_node. 0 berarti fitur node dinolkan (absen)."""
    x = data.x.clone()
    x[mask == 0] = 0.0
    d = Data(x=x, edge_index=data.edge_index, batch=torch.zeros(x.size(0), dtype=torch.long)).to(DEV)
    probs = [torch.softmax(m(d), dim=1)[0, 1].item() for m in models]
    return np.mean(probs)

def shapley_exact(models, data):
    n = data.num_nodes
    nilai = np.zeros(n)
    semua_node = list(range(n))
    for i in range(n):
        lainnya = [j for j in semua_node if j != i]
        total = 0.0
        for r in range(len(lainnya) + 1):
            for subset in itertools.combinations(lainnya, r):
                subset = set(subset)
                bobot = factorial(r) * factorial(len(lainnya) - r) / factorial(len(lainnya) + 1)
                mask_tanpa = np.zeros(n);
                for j in subset: mask_tanpa[j] = 1
                mask_dengan = mask_tanpa.copy(); mask_dengan[i] = 1
                v_tanpa = prob_rentan_masked(models, data, mask_tanpa)
                v_dengan = prob_rentan_masked(models, data, mask_dengan)
                total += bobot * (v_dengan - v_tanpa)
        nilai[i] = total
    return nilai

print('fungsi Shapley eksak siap')

fungsi Shapley eksak siap


In [ ]:
GROUND_TRUTH = lambda n: (n['n_call_low']+n['n_call_high']>0) or (n['n_sstore']>0)

hasil_shap = {}
mulai = time.time()
for d in data_bench:
    if d.num_nodes > 12:
        print(f'{d.nama} dilewati (n={d.num_nodes} > 12, terlalu besar untuk eksak)')
        continue
    nilai = shapley_exact(models_mp, d)   # pakai model mean pooling
    hasil_shap[d.nama] = nilai
    print(f'{d.nama:32} n={d.num_nodes}  selesai ({(time.time()-mulai):.0f} dtk berjalan)')

print(f'\ntotal waktu: {(time.time()-mulai)/60:.1f} menit')

etherbank_fixed.sol              n=7  selesai (19 dtk berjalan)
etherstore_fixed.sol             n=10  selesai (233 dtk berjalan)
modifier_reentrancy_fixed.sol    n=7  selesai (251 dtk berjalan)
reentrance_fixed.sol             n=10  selesai (462 dtk berjalan)
reentrancy_dao_fixed.sol         n=9  selesai (556 dtk berjalan)
reentrancy_insecure_fixed.sol    n=6  selesai (563 dtk berjalan)
reentrancy_simple_fixed.sol      n=7  selesai (581 dtk berjalan)
simple_dao_fixed.sol             n=7  selesai (600 dtk berjalan)
etherbank.sol                    n=7  selesai (617 dtk berjalan)
etherstore.sol                   n=10  selesai (830 dtk berjalan)
modifier_reentrancy.sol dilewati (n=14 > 12, terlalu besar untuk eksak)
reentrance.sol                   n=10  selesai (1041 dtk berjalan)
reentrancy_dao.sol               n=9  selesai (1134 dtk berjalan)
reentrancy_insecure.sol          n=6  selesai (1142 dtk berjalan)
reentrancy_simple.sol            n=7  selesai (1160 dtk berjalan)
simple_dao.

In [ ]:
OUT_PRUNE_BM = f'{D}/data/processed/v2_cfg_prune_0.02'  # sumber JSON asli untuk baca atribut node

def baca_json(nama_file, folder):
    fn = f"{folder}__{nama_file[:-4]}.json"
    with open(f'{OUT_PRUNE_BM}/{fn}') as f:
        return json.load(f)

for nama_pt, nilai in hasil_shap.items():
    if 'fixed' in nama_pt:
        folder = '3_expert_benchmark__aman'
    else:
        folder = '3_expert_benchmark__rentan'
    g = baca_json(nama_pt, folder)
    nodes = sorted(g['nodes'], key=lambda x: x['id'])
    print(f'\n=== {nama_pt} ===')
    urutan = np.argsort(-nilai)
    for idx in urutan:
        n = nodes[idx]
        gt = 'CALL/SSTORE' if GROUND_TRUTH(n) else ''
        print(f'  node {idx:2} shap={nilai[idx]:+.4f}  {n["tipe"]:10} {gt}')


=== etherbank_fixed.sol ===
  node  5 shap=+0.0003  IF         CALL/SSTORE
  node  4 shap=+0.0003  EXPRESSION CALL/SSTORE
  node  3 shap=+0.0003  ENTRYPOINT 
  node  1 shap=+0.0003  ENTRYPOINT 
  node  6 shap=+0.0003  THROW      
  node  2 shap=+0.0002  EXPRESSION CALL/SSTORE
  node  0 shap=+0.0002  ENTRYPOINT 

=== etherstore_fixed.sol ===
  node  9 shap=+0.0002  OTHER_ENTRYPOINT CALL/SSTORE
  node  3 shap=+0.0001  EXPRESSION 
  node  5 shap=+0.0001  EXPRESSION 
  node  7 shap=+0.0001  EXPRESSION CALL/SSTORE
  node  4 shap=+0.0001  EXPRESSION 
  node  6 shap=+0.0001  EXPRESSION CALL/SSTORE
  node  8 shap=+0.0001  EXPRESSION CALL/SSTORE
  node  1 shap=+0.0000  EXPRESSION CALL/SSTORE
  node  0 shap=-0.0001  ENTRYPOINT 
  node  2 shap=-0.0001  ENTRYPOINT 

=== modifier_reentrancy_fixed.sol ===
  node  4 shap=+0.0005  OTHER_ENTRYPOINT CALL/SSTORE
  node  3 shap=+0.0005  EXPRESSION 
  node  2 shap=+0.0005  EXPRESSION CALL/SSTORE
  node  1 shap=+0.0004  EXPRESSION CALL/SSTORE
  node  6 sha

In [ ]:
# klasifikasi kontrak: flat vs discriminative attribution
print(f"{'kontrak':32} {'std_shap':>10} {'top1_gt':>8} {'pola':>14}")
for nama, nilai in hasil_shap.items():
    std_shap = np.std(nilai)
    idx_top1 = np.argmax(nilai)
    folder = '3_expert_benchmark__aman' if 'fixed' in nama else '3_expert_benchmark__rentan'
    g = baca_json(nama, folder)
    nodes = sorted(g['nodes'], key=lambda x: x['id'])
    top1_gt = GROUND_TRUTH(nodes[idx_top1])
    pola = 'flat' if std_shap < 0.001 else 'discriminative'
    print(f"{nama:32} {std_shap:10.5f} {str(top1_gt):>8} {pola:>14}")

kontrak                            std_shap  top1_gt           pola
etherbank_fixed.sol                 0.00003     True           flat
etherstore_fixed.sol                0.00008     True           flat
modifier_reentrancy_fixed.sol       0.00018     True           flat
reentrance_fixed.sol                0.01058    False discriminative
reentrancy_dao_fixed.sol            0.02805    False discriminative
reentrancy_insecure_fixed.sol       0.01955     True discriminative
reentrancy_simple_fixed.sol         0.00003     True           flat
simple_dao_fixed.sol                0.00147    False discriminative
etherbank.sol                       0.00002     True           flat
etherstore.sol                      0.00007     True           flat
reentrance.sol                      0.00623    False discriminative
reentrancy_dao.sol                  0.04418    False discriminative
reentrancy_insecure.sol             0.05626    False discriminative
reentrancy_simple.sol               0.00014     

In [ ]:
def iou_shap_groundtruth(nilai, nodes, k=2):
    top_k = set(np.argsort(-nilai)[:k])
    gt = {n['id'] for n in nodes if GROUND_TRUTH(n)}
    if not gt:
        return None
    inter = len(top_k & gt)
    union = len(top_k | gt)
    return inter / union if union else 0

print(f"\n{'kontrak':32} {'IoU (k=2)':>10}")
skor_iou = []
for nama, nilai in hasil_shap.items():
    folder = '3_expert_benchmark__aman' if 'fixed' in nama else '3_expert_benchmark__rentan'
    g = baca_json(nama, folder)
    nodes = sorted(g['nodes'], key=lambda x: x['id'])
    iou = iou_shap_groundtruth(nilai, nodes)
    if iou is not None:
        skor_iou.append(iou)
        print(f"{nama:32} {iou:10.3f}")

print(f"\nrata-rata IoU: {np.mean(skor_iou):.3f}")


kontrak                           IoU (k=2)
etherbank_fixed.sol                   0.667
etherstore_fixed.sol                  0.167
modifier_reentrancy_fixed.sol         0.250
reentrance_fixed.sol                  0.000
reentrancy_dao_fixed.sol              0.167
reentrancy_insecure_fixed.sol         0.333
reentrancy_simple_fixed.sol           0.667
simple_dao_fixed.sol                  0.250
etherbank.sol                         0.250
etherstore.sol                        0.400
reentrance.sol                        0.000
reentrancy_dao.sol                    0.167
reentrancy_insecure.sol               0.333
reentrancy_simple.sol                 0.250
simple_dao.sol                        0.250

rata-rata IoU: 0.277
